<a href="https://colab.research.google.com/github/Saifullah785/machine-learning-engineer-roadmap/blob/main/Lecture_78_Optuna_Basics_yt/Lecture_78_Optuna_Basics_yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 10.8 MB/s eta 0:00:00


In [2]:
# import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
# load the pima indian diabetes dataset from sklearn
# Note: scikit-learn's build-in 'load_diabetes' is a regression dataset.
# we will load the actual diabetes dataset from an external source

import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [4]:
# load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
import numpy as np

# Replace zero values with NaN is columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

#check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Define the objective function
def objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators',  50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # create the randomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )
  # Perform 3-fold cross-validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score

In [8]:
# create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler()) # we aim to maximize accuracy

# RUn 50 trials to find the best hyperparameters
study.optimize(objective, n_trials=50)

[I 2025-09-11 07:39:17,976] A new study created in memory with name: no-name-fb20f82b-5913-4180-8641-04ae4a09f1d3
[I 2025-09-11 07:39:19,704] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 187, 'max_depth': 11}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-09-11 07:39:22,460] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 156, 'max_depth': 9}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-09-11 07:39:23,608] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 91, 'max_depth': 12}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-09-11 07:39:24,620] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 83, 'max_depth': 7}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-09-11 07:39:27,081] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 185, 'max_depth': 17}. Best is trial 0 with value: 0.77094972

In [9]:
# print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 119, 'max_depth': 20}


In [10]:
from sklearn.metrics import accuracy_score

#  Train a RandomForestClassifier using the best hyperparameters from optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# print the test acccuracy

print(f'Test accuracy with best hyperparameters: {test_accuracy:.2f}')

Test accuracy with best hyperparameters: 0.75


## Optuna Visualizations

In [13]:
# For visualizations

from optuna.visualization import plot_optimization_history, \
plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [14]:
# 1. Optimization History
plot_optimization_history(study).show()

In [15]:
# 2. parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [16]:
# 3.Slice Plot
plot_slice(study).show()

In [17]:
# 4. Contour Plot
plot_contour(study).show()

In [18]:
# 5. Parameter Importances Plot
plot_param_importances(study).show()